In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import shapiro, friedmanchisquare
import os
import warnings
warnings.filterwarnings('ignore')

# Configuration
MODELS = {
    'deepseek-r1_7b': 'DeepSeek R1 7B',
    'gemma3_4b': 'Gemma3 4B', 
    'granite3.1-moe_3b': 'Granite 3.1 MoE 3B',
    'llama3.1_8b-instruct-q3_K_M': 'Llama 3.1 8B',
    'mistral_7b-instruct': 'Mistral 7B',
    'qwen3_4b-instruct': 'Qwen3 4B'
}

CSV_FILES = {
    'deepseek-r1_7b': 'evaluation_summary_with_review_deepseek-r1_7b_6tasks.csv',
    'gemma3_4b': 'evaluation_summary_with_review_gemma3_4b_6tasks.csv',
    'granite3.1-moe_3b': 'evaluation_summary_with_review_granite3.1-moe_3b_6tasks.csv',
    'llama3.1_8b-instruct-q3_K_M': 'evaluation_summary_with_review_llama3.1_8b-instruct-q3_K_M_6tasks.csv',
    'mistral_7b-instruct': 'evaluation_summary_with_review_mistral_7b-instruct_6tasks.csv',
    'qwen3_4b-instruct': 'evaluation_summary_with_review_qwen3_4b-instruct_6tasks.csv'
}

BASE_DIR = "../Output Performance Degradation Analysis"
RESULTS_DIR = "Statistical_Validation_Results"

def create_results_directory():
    """Create dedicated directory for statistical results."""
    results_path = os.path.join(BASE_DIR, RESULTS_DIR)
    os.makedirs(results_path, exist_ok=True)
    return results_path

def load_data():
    """Load CSV data for all models."""
    all_data = {}
    
    for model_key, model_name in MODELS.items():
        csv_file = CSV_FILES.get(model_key)
        csv_path = os.path.join(BASE_DIR, csv_file)
        
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                all_data[model_key] = df
                print(f"Loaded: {model_name} ({len(df)} rows)")
            except Exception as e:
                print(f"Error loading {model_name}: {e}")
        else:
            print(f"File not found: {csv_path}")
    
    return all_data

class StatisticalValidator:
    """Simplified statistical validation with only Shapiro-Wilk and Friedman tests."""
    
    def __init__(self, data):
        self.data = data
        self._analyze_data_structure()
        
    def _analyze_data_structure(self):
        """Analyze available metrics for Task 6."""
        task6_metrics = set()
        
        for model_key, df in self.data.items():
            task6_data = df[df['task_number'] == 6]
            for col in ['bleu_review', 'bleu_translate', 'f1_ner', 'json_em', 'em_sentiment', 'em_emotion', 'em_topic']:
                if col in df.columns and task6_data[col].notna().sum() > 0:
                    task6_metrics.add(col)
        
        self.test_metrics = list(task6_metrics)
        print(f"Testing metrics: {self.test_metrics}")
        
    def test_shapiro_wilk(self):
        """Shapiro-Wilk test for normality - simplified results."""
        results = []
        
        for model_key, df in self.data.items():
            model_name = MODELS[model_key]
            
            for metric in self.test_metrics:
                if metric in df.columns:
                    values = df[metric].dropna()
                    
                    if len(values) >= 3:
                        try:
                            stat, p_value = shapiro(values)
                            
                            results.append({
                                'Model': model_name,
                                'Metric': metric,
                                'P_Value': round(p_value, 4)
                            })
                            
                        except Exception as e:
                            continue
        
        return pd.DataFrame(results)
    
    def test_friedman(self):
        """Friedman test for multi-model comparison - simplified results."""
        results = []
        
        for metric in self.test_metrics:
            # Collect Task 6 performance for each model
            model_performances = {}
            
            for model_key, df in self.data.items():
                model_name = MODELS[model_key]
                task6_data = df[df['task_number'] == 6][metric].dropna()
                
                if not task6_data.empty:
                    model_performances[model_name] = task6_data.values
            
            if len(model_performances) >= 3:
                try:
                    performance_lists = list(model_performances.values())
                    friedman_stat, p_value = friedmanchisquare(*performance_lists)
                    
                    # Calculate model medians for ranking
                    model_medians = {name: round(np.median(values), 3) 
                                   for name, values in model_performances.items()}
                    
                    best_model = max(model_medians.items(), key=lambda x: x[1])
                    
                    results.append({
                        'Metric': metric,
                        'P_Value': round(p_value, 4),
                        'Best_Model': best_model[0]
                    })
                    
                except Exception as e:
                    continue
        
        return pd.DataFrame(results)
    
    def run_tests(self):
        """Run simplified statistical tests."""
        print("Running Shapiro-Wilk normality test...")
        shapiro_results = self.test_shapiro_wilk()
        
        print("Running Friedman test...")
        friedman_results = self.test_friedman()
        
        return {
            'shapiro_wilk': shapiro_results,
            'friedman': friedman_results
        }

def create_simple_table(df, title, filename, results_dir):
    """Create simple statistical tables matching the minimal style."""
    if df.empty:
        print(f"No data for {title}")
        return
    
    # Calculate figure size based on content - add more space for title
    fig_height = max(4.0, len(df) * 0.3 + 2.5)  # Increased base height further
    fig_width = max(8, len(df.columns) * 1.5)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    
    # Create table with equal column widths - position it lower to make room for title
    col_width = 0.8 / len(df.columns)
    table = ax.table(cellText=df.values,
                    colLabels=df.columns,
                    cellLoc='center',
                    loc='center',
                    colWidths=[col_width] * len(df.columns))
    
    # Minimal styling
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.6)  # Reduced scale to make more room
    
    # Simple borders like the example
    for key, cell in table.get_celld().items():
        cell.set_linewidth(1)
        cell.set_edgecolor('black')
        cell.set_facecolor('white')
        
        row, col = key
        if row == 0:  # Header
            cell.set_text_props(weight='bold')
    
    # Position title with more space above
    plt.suptitle(title, fontsize=12, fontweight='bold', y=0.95, x=0.5, ha='center')
    
    # Adjust layout to prevent overlap - more top margin
    plt.subplots_adjust(top=0.88, bottom=0.05)
    
    output_path = os.path.join(results_dir, filename)
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

def assess_validity(results):
    """Simple validity assessment."""
    print("\n" + "="*50)
    print("STATISTICAL VALIDITY ASSESSMENT")
    print("="*50)
    
    # Normality assessment
    if not results['shapiro_wilk'].empty:
        total_tests = len(results['shapiro_wilk'])
        normal_count = (results['shapiro_wilk']['P_Value'] > 0.05).sum()
        normal_pct = (normal_count / total_tests) * 100
        
        print(f"Normality: {normal_count}/{total_tests} ({normal_pct:.1f}%) normal distributions")
        print("→ Non-parametric tests justified" if normal_pct < 50 else "→ Mixed distribution types")
    
    # Model differences
    if not results['friedman'].empty:
        significant_diffs = (results['friedman']['P_Value'] < 0.05).sum()
        total_metrics = len(results['friedman'])
        sig_pct = (significant_diffs / total_metrics) * 100
        
        print(f"Model differences: {significant_diffs}/{total_metrics} ({sig_pct:.1f}%) metrics significant")
        
        if significant_diffs > 0:
            print("\nBest performing models:")
            sig_results = results['friedman'][results['friedman']['P_Value'] < 0.05]
            for _, row in sig_results.iterrows():
                print(f"  {row['Metric']}: {row['Best_Model']}")
    
    # Overall validity
    validity_score = 0
    if normal_pct < 50: validity_score += 1
    if sig_pct >= 50: validity_score += 1
    if total_metrics >= 5: validity_score += 1
    
    if validity_score >= 2:
        assessment = "HIGH VALIDITY - Results statistically reliable"
    elif validity_score == 1:
        assessment = "MODERATE VALIDITY - Results reasonably reliable"
    else:
        assessment = "LIMITED VALIDITY - Results require caution"
    
    print(f"\nFinal Assessment: {assessment}")
    return assessment

def main():
    """Main execution function."""
    print("SIMPLIFIED STATISTICAL VALIDATION")
    print("Shapiro-Wilk + Friedman Tests Only")
    print("="*50)
    
    # Create results directory
    results_dir = create_results_directory()
    print(f"Results directory: {results_dir}")
    
    # Load data
    all_data = load_data()
    if not all_data:
        print("No data loaded. Exiting.")
        return
    
    # Run tests
    validator = StatisticalValidator(all_data)
    results = validator.run_tests()
    
    # Create tables
    print("\nCreating result tables...")
    
    create_simple_table(
        results['shapiro_wilk'], 
        'Shapiro-Wilk Normality Test Results',
        'shapiro_wilk_results.png',
        results_dir
    )
    
    create_simple_table(
        results['friedman'], 
        'Friedman Test Results (Multi-Model Comparison)',
        'friedman_results.png',
        results_dir
    )
    
    # Validity assessment
    assessment = assess_validity(results)
    
    print(f"\nAll results saved to: {results_dir}")
    return results, assessment

if __name__ == "__main__":
    main()

SIMPLIFIED STATISTICAL VALIDATION
Shapiro-Wilk + Friedman Tests Only
Results directory: ../Output Performance Degradation Analysis\Statistical_Validation_Results
Loaded: DeepSeek R1 7B (21 rows)
Loaded: Gemma3 4B (21 rows)
Loaded: Granite 3.1 MoE 3B (21 rows)
Loaded: Llama 3.1 8B (21 rows)
Loaded: Mistral 7B (21 rows)
Loaded: Qwen3 4B (21 rows)
Testing metrics: ['json_em', 'em_emotion', 'bleu_translate', 'bleu_review', 'em_sentiment', 'f1_ner', 'em_topic']
Running Shapiro-Wilk normality test...
Running Friedman test...

Creating result tables...
Saved: shapiro_wilk_results.png
Saved: friedman_results.png

STATISTICAL VALIDITY ASSESSMENT
Normality: 5/36 (13.9%) normal distributions
→ Non-parametric tests justified
Model differences: 5/7 (71.4%) metrics significant

Best performing models:
  json_em: Qwen3 4B
  em_emotion: Qwen3 4B
  bleu_translate: Gemma3 4B
  bleu_review: Gemma3 4B
  em_sentiment: Mistral 7B

Final Assessment: HIGH VALIDITY - Results statistically reliable

All results

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import shapiro, friedmanchisquare
import os
import warnings
warnings.filterwarnings('ignore')

# Configuration
MODELS = {
    'deepseek-r1_7b': 'DeepSeek R1 7B',
    'gemma3_4b': 'Gemma3 4B', 
    'granite3.1-moe_3b': 'Granite 3.1 MoE 3B',
    'llama3.1_8b-instruct-q3_K_M': 'Llama 3.1 8B',
    'mistral_7b-instruct': 'Mistral 7B',
    'qwen3_4b-instruct': 'Qwen3 4B'
}

CSV_FILES = {
    'd
    eepseek-r1_7b': 'evaluation_summary_with_review_deepseek-r1_7b_6tasks.csv',
    'gemma3_4b': 'evaluation_summary_with_review_gemma3_4b_6tasks.csv',
    'granite3.1-moe_3b': 'evaluation_summary_with_review_granite3.1-moe_3b_6tasks.csv',
    'llama3.1_8b-instruct-q3_K_M': 'evaluation_summary_with_review_llama3.1_8b-instruct-q3_K_M_6tasks.csv',
    'mistral_7b-instruct': 'evaluation_summary_with_review_mistral_7b-instruct_6tasks.csv',
    'qwen3_4b-instruct': 'evaluation_summary_with_review_qwen3_4b-instruct_6tasks.csv'
}

BASE_DIR = "../Output Performance Degradation Analysis"
RESULTS_DIR = "Statistical_Validation_Results"

def create_results_directory():
    """Create dedicated directory for statistical results."""
    results_path = os.path.join(BASE_DIR, RESULTS_DIR)
    os.makedirs(results_path, exist_ok=True)
    return results_path

def load_data():
    """Load CSV data for all models."""
    all_data = {}
    
    for model_key, model_name in MODELS.items():
        csv_file = CSV_FILES.get(model_key)
        csv_path = os.path.join(BASE_DIR, csv_file)
        
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                all_data[model_key] = df
                print(f"Loaded: {model_name} ({len(df)} rows)")
            except Exception as e:
                print(f"Error loading {model_name}: {e}")
        else:
            print(f"File not found: {csv_path}")
    
    return all_data

class StatisticalValidator:
    """Simplified statistical validation with only Shapiro-Wilk and Friedman tests."""
    
    def __init__(self, data):
        self.data = data
        self._analyze_data_structure()
        
    def _analyze_data_structure(self):
        """Analyze available metrics for Task 6."""
        task6_metrics = set()
        
        print("\n" + "="*50)
        print("ANALISI DISPONIBILITÀ DATI PER TASK 6")
        print("="*50)
        
        for model_key, df in self.data.items():
            model_name = MODELS[model_key]
            task6_data = df[df['task_number'] == 6]
            
            print(f"\n{model_name}:")
            for col in ['bleu_review', 'bleu_translate', 'f1_ner', 'json_em', 'em_sentiment', 'em_emotion', 'em_topic']:
                if col in df.columns:
                    valid_count = task6_data[col].notna().sum()
                    print(f"  {col}: {valid_count} valori validi")
                    if valid_count > 0:
                        task6_metrics.add(col)
                else:
                    print(f"  {col}: colonna NON presente")
        
        self.test_metrics = list(task6_metrics)
        print(f"\n{'='*50}")
        print(f"Metriche che verranno testate: {self.test_metrics}")
        print("="*50)
        
    def test_shapiro_wilk(self):
        """Shapiro-Wilk test for normality - simplified results with debugging."""
        results = []
        
        print("\n" + "="*50)
        print("DEBUG SHAPIRO-WILK TEST")
        print("="*50)
        
        for model_key, df in self.data.items():
            model_name = MODELS[model_key]
            
            for metric in self.test_metrics:
                if metric in df.columns:
                    values = df[metric].dropna()
                    
                    print(f"\n{model_name} - {metric}:")
                    print(f"  Valori totali: {len(values)}")
                    
                    if len(values) >= 3:
                        try:
                            stat, p_value = shapiro(values)
                            
                            results.append({
                                'Model': model_name,
                                'Metric': metric,
                                'P_Value': round(p_value, 4)
                            })
                            print(f"  ✓ Test eseguito con successo (p={p_value:.4f})")
                            
                        except Exception as e:
                            print(f"  ✗ ERRORE nel test: {e}")
                            continue
                    else:
                        print(f"  ✗ SALTATO: dati insufficienti (min 3 richiesti)")
        
        return pd.DataFrame(results)
    
    def test_friedman(self):
        """Friedman test for multi-model comparison - simplified results with debugging."""
        results = []
        
        print("\n" + "="*50)
        print("DEBUG FRIEDMAN TEST")
        print("="*50)
        
        for metric in self.test_metrics:
            print(f"\nMetrica: {metric}")
            
            # Collect Task 6 performance for each model
            model_performances = {}
            
            for model_key, df in self.data.items():
                model_name = MODELS[model_key]
                task6_data = df[df['task_number'] == 6][metric].dropna()
                
                if not task6_data.empty:
                    model_performances[model_name] = task6_data.values
                    print(f"  {model_name}: {len(task6_data.values)} valori")
            
            print(f"  Modelli con dati: {len(model_performances)}")
            
            if len(model_performances) >= 3:
                try:
                    performance_lists = list(model_performances.values())
                    friedman_stat, p_value = friedmanchisquare(*performance_lists)
                    
                    # Calculate model medians for ranking
                    model_medians = {name: round(np.median(values), 3) 
                                   for name, values in model_performances.items()}
                    
                    best_model = max(model_medians.items(), key=lambda x: x[1])
                    
                    results.append({
                        'Metric': metric,
                        'P_Value': round(p_value, 4),
                        'Best_Model': best_model[0]
                    })
                    print(f"  ✓ Test eseguito (p={p_value:.4f}, best={best_model[0]})")
                    
                except Exception as e:
                    print(f"  ✗ ERRORE nel test: {e}")
                    continue
            else:
                print(f"  ✗ SALTATO: modelli insufficienti (min 3 richiesti)")
        
        return pd.DataFrame(results)
    
    def run_tests(self):
        """Run simplified statistical tests."""
        print("\nRunning Shapiro-Wilk normality test...")
        shapiro_results = self.test_shapiro_wilk()
        
        print("\n\nRunning Friedman test...")
        friedman_results = self.test_friedman()
        
        return {
            'shapiro_wilk': shapiro_results,
            'friedman': friedman_results
        }

def create_simple_table(df, title, filename, results_dir):
    """Create simple statistical tables matching the minimal style."""
    if df.empty:
        print(f"No data for {title}")
        return
    
    # Calculate figure size based on content - add more space for title
    fig_height = max(4.0, len(df) * 0.3 + 2.5)  # Increased base height further
    fig_width = max(8, len(df.columns) * 1.5)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    
    # Create table with equal column widths - position it lower to make room for title
    col_width = 0.8 / len(df.columns)
    table = ax.table(cellText=df.values,
                    colLabels=df.columns,
                    cellLoc='center',
                    loc='center',
                    colWidths=[col_width] * len(df.columns))
    
    # Minimal styling
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.6)  # Reduced scale to make more room
    
    # Simple borders like the example
    for key, cell in table.get_celld().items():
        cell.set_linewidth(1)
        cell.set_edgecolor('black')
        cell.set_facecolor('white')
        
        row, col = key
        if row == 0:  # Header
            cell.set_text_props(weight='bold')
    
    # Position title with more space above
    plt.suptitle(title, fontsize=12, fontweight='bold', y=0.95, x=0.5, ha='center')
    
    # Adjust layout to prevent overlap - more top margin
    plt.subplots_adjust(top=0.88, bottom=0.05)
    
    output_path = os.path.join(results_dir, filename)
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

def assess_validity(results):
    """Simple validity assessment."""
    print("\n" + "="*50)
    print("STATISTICAL VALIDITY ASSESSMENT")
    print("="*50)
    
    # Normality assessment
    if not results['shapiro_wilk'].empty:
        total_tests = len(results['shapiro_wilk'])
        normal_count = (results['shapiro_wilk']['P_Value'] > 0.05).sum()
        normal_pct = (normal_count / total_tests) * 100
        
        print(f"Normality: {normal_count}/{total_tests} ({normal_pct:.1f}%) normal distributions")
        print("→ Non-parametric tests justified" if normal_pct < 50 else "→ Mixed distribution types")
    
    # Model differences
    if not results['friedman'].empty:
        significant_diffs = (results['friedman']['P_Value'] < 0.05).sum()
        total_metrics = len(results['friedman'])
        sig_pct = (significant_diffs / total_metrics) * 100
        
        print(f"Model differences: {significant_diffs}/{total_metrics} ({sig_pct:.1f}%) metrics significant")
        
        if significant_diffs > 0:
            print("\nBest performing models:")
            sig_results = results['friedman'][results['friedman']['P_Value'] < 0.05]
            for _, row in sig_results.iterrows():
                print(f"  {row['Metric']}: {row['Best_Model']}")
    
    # Overall validity
    validity_score = 0
    if normal_pct < 50: validity_score += 1
    if sig_pct >= 50: validity_score += 1
    if total_metrics >= 5: validity_score += 1
    
    if validity_score >= 2:
        assessment = "HIGH VALIDITY - Results statistically reliable"
    elif validity_score == 1:
        assessment = "MODERATE VALIDITY - Results reasonably reliable"
    else:
        assessment = "LIMITED VALIDITY - Results require caution"
    
    print(f"\nFinal Assessment: {assessment}")
    return assessment

def main():
    """Main execution function."""
    print("SIMPLIFIED STATISTICAL VALIDATION - WITH DEBUG")
    print("Shapiro-Wilk + Friedman Tests Only")
    print("="*50)
    
    # Create results directory
    results_dir = create_results_directory()
    print(f"Results directory: {results_dir}")
    
    # Load data
    all_data = load_data()
    if not all_data:
        print("No data loaded. Exiting.")
        return
    
    # Run tests
    validator = StatisticalValidator(all_data)
    results = validator.run_tests()
    
    # Create tables
    print("\n\nCreating result tables...")
    
    create_simple_table(
        results['shapiro_wilk'], 
        'Shapiro-Wilk Normality Test Results',
        'shapiro_wilk_results.png',
        results_dir
    )
    
    create_simple_table(
        results['friedman'], 
        'Friedman Test Results (Multi-Model Comparison)',
        'friedman_results.png',
        results_dir
    )
    
    # Validity assessment
    assessment = assess_validity(results)
    
    print(f"\nAll results saved to: {results_dir}")
    return results, assessment

if __name__ == "__main__":
    main()

SIMPLIFIED STATISTICAL VALIDATION - WITH DEBUG
Shapiro-Wilk + Friedman Tests Only
Results directory: ../Output Performance Degradation Analysis\Statistical_Validation_Results
Loaded: DeepSeek R1 7B (21 rows)
Loaded: Gemma3 4B (21 rows)
Loaded: Granite 3.1 MoE 3B (21 rows)
Loaded: Llama 3.1 8B (21 rows)
Loaded: Mistral 7B (21 rows)
Loaded: Qwen3 4B (21 rows)

ANALISI DISPONIBILITÀ DATI PER TASK 6

DeepSeek R1 7B:
  bleu_review: 6 valori validi
  bleu_translate: 5 valori validi
  f1_ner: 1 valori validi
  json_em: 6 valori validi
  em_sentiment: 4 valori validi
  em_emotion: 3 valori validi
  em_topic: 2 valori validi

Gemma3 4B:
  bleu_review: 6 valori validi
  bleu_translate: 5 valori validi
  f1_ner: 1 valori validi
  json_em: 6 valori validi
  em_sentiment: 4 valori validi
  em_emotion: 3 valori validi
  em_topic: 2 valori validi

Granite 3.1 MoE 3B:
  bleu_review: 6 valori validi
  bleu_translate: 5 valori validi
  f1_ner: 1 valori validi
  json_em: 6 valori validi
  em_sentiment: 4